# Label head dataset for answers expecting time unit "year"

## Load data

In [1]:
import pandas as pd 

In [2]:
head_df = pd.read_csv("./data/notebook_output/02_man_label_temp_q_head_data_clean.csv")
head_df.head()

,question,answer,table_id,answer_type,is_temporal,answer_old
0,How many years did Art Carney as actor since 1...,54 Years,2,TEMPORAL,Is temporal,54 Years
1,How many total years was Art Carney married to...,28 years,2,TEMPORAL,Is temporal,28 years
2,How many years before he died was Art Carney m...,23,2,COUNT,Is temporal,23
3,How old was Art Carney when he first got divor...,47,2,AGE,Is temporal,47
4,How many years ago did Art Carney was died?,19 Years ago,2,TEMPORAL,Is temporal,19 Years ago


## Identify "year" time unit from answer

In [3]:
# Label most obvious QA-pairs where year is in answer
# But, answers can be mixed, e.g., "1 year 2 months"
# Need to filter them
head_df.query("answer.str.lower().str.contains('year')").tail()

,question,answer,table_id,answer_type,is_temporal,answer_old
1109,How long did it take for Sri Lanka to become a...,16 years,1191,TEMPORAL,Is temporal,16 years
1112,How long after Sri Lanka's first ODI and first...,31 years,1191,TEMPORAL,Is temporal,31 years
1113,What is the time span for World Test Champions...,2 year,1191,TEMPORAL,Is temporal,2 year
1119,How many years did the Ingenuity fly?,1 year,1206,TEMPORAL,Is temporal,1 year
1121,How much time since deployment did it take for...,1 year 2 months,1206,TEMPORAL,Is temporal,1 year 2 months


### Answer containing \d+ year(s) (ago|after|old)

In [4]:
# Pre-select where answer has time unit "year"
# Find false-positives by excluding those that certainly have time unit "year"
pot_false_pos = head_df.query("answer.str.lower().str.contains('year')").query(
    "~answer.str.lower().str.match('\d+ years?(?:\s+ago)?(?:\s+old)?(?:\s+after)?\.?$')"
)
pot_false_pos

,question,answer,table_id,answer_type,is_temporal,answer_old
22,How many years did Joan Crawford was active ca...,50 Years (1924–1974),8,TEMPORAL,Is temporal,50 Years (1924–1974)
23,How many years did Joan Crawford and Alfred St...,4 Years (from 1955 to 1959),8,TEMPORAL,Is temporal,4 Years (from 1955 to 1959)
39,How many years did Antonio Mohamed was played ...,3 Years (1989-1991),36,TEMPORAL,Is temporal,3 Years (1989-1991)
40,How many years did Zinedine Zidane was played ...,17 Years played,40,TEMPORAL,Is temporal,17 Years played
98,How many years ago did Nate Archibald was rece...,16 Years ago (2006),83,TEMPORAL,Is temporal,16 Years ago (2006)
107,How many years ago did Scottie Pippen awarded ...,27 Years ago (1995),89,TEMPORAL,Is temporal,27 Years ago (1995)
116,How many years did Thiago Alves played for Grê...,3 Years (2000-2003),96,TEMPORAL,Is temporal,3 Years (2000-2003)
139,How many years ago did Shi Yuqi achived highes...,5 Years ago (2017),166,TEMPORAL,Is temporal,5 Years ago (2017)
187,How many years ago did Casey Mears last partic...,21 Years ago (2001),264,TEMPORAL,Is temporal,21 Years ago (2001)
188,How many years ago did Casey Mears received th...,15 Years ago (2007),264,TEMPORAL,Is temporal,15 Years ago (2007)


Most potential false positives follow the format of answering the question with the number of years but include _the_ year(s) as well in parenthesis.

I will deal with those later. Let us look closer which other types of potential false-positives are there.

In [5]:
pot_false_pos.query("~answer.str.contains('\d{4}\)')")

,question,answer,table_id,answer_type,is_temporal,answer_old
40,How many years did Zinedine Zidane was played ...,17 Years played,40,TEMPORAL,Is temporal,17 Years played
584,What was the time difference between the first...,Almost 22 years,629,TEMPORAL,Is temporal,Almost 22 years
585,What was the time difference between the first...,Almost 14 years,629,TEMPORAL,Is temporal,Almost 14 years
685,How many total years (terms) does Joe Biden ho...,6 years (2 terms),743,TEMPORAL,Is temporal,6 years (2 terms)
763,What was the time.between Music's ordination a...,Nearly 30 years,824,TEMPORAL,Is temporal,Nearly 30 years
764,How long was Music's bishop term for?,Nearly 30 years,824,TEMPORAL,Is temporal,Nearly 30 years
1009,How long after launch was INSTrishulentering J...,1 year and 7 months.,1113,TEMPORAL,Is temporal,1 year and 7 months.
1010,How long after INSTrishulentering Jubail was o...,5 years 7 months,1113,TEMPORAL,Is temporal,5 years 7 months
1043,How many years completed that V/Line Corporati...,19 Years completed,1141,TEMPORAL,Is temporal,19 Years completed
1051,How many years did 5th edition released after ...,20 Years after first edition,1147,TEMPORAL,Is temporal,20 Years after first edition


The remaining potential false positives include a mix of years and months, imprecise dates (nearly/almost), or are true positives but include a verb at the end (played/completed). 

Thus, I will extend my first regex to also include "played" and "completed" and keep it otherwise as is.

In [6]:
head_df.loc[:, "answer_timeunit"] = "not defined"

In [7]:
def label_timeunit_year(df: pd.DataFrame, index_to_label: pd.Index) -> pd.DataFrame:
    df.loc[index_to_label, "answer_timeunit"] = "year"
    return df

In [8]:
head_df = label_timeunit_year(
    head_df,
    head_df.query(
        "answer.str.lower().str.contains('year') and "
        "answer.str.lower().str.match('\d+ years?(?:\s+ago)?(?:\s+old)?(?:\s+after)?(?:\s+plaed)?(?:\s+completed)?\.?$')"
    ).index,
)

### Answer containing Age of \d+ 

In [9]:
head_df.query(
    "answer.str.lower().str.match('age of \d+$')"
)

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit
16,What was the age when Dwayne Johnson as debut ...,Age of 27,7,AGE,Is temporal,Age of 27,not defined
25,What was the the age when Hebar Pazardzhik ori...,Age of 39,13,UNKNOWN,Is temporal,Age of 39,not defined
102,What was the age when Scottie Pippen won gold ...,Age of 31,89,AGE,Is temporal,Age of 31,not defined
113,What was the age when Thiago Alves won first m...,Age of 26,96,AGE,Is temporal,Age of 26,not defined
149,What was the age when Wang Xiaoli won first go...,Age of 17,174,AGE,Is temporal,Age of 17,not defined
196,What was the age when Cody Ware participated f...,Age of 19,265,AGE,Is temporal,Age of 19,not defined
200,What was the age when Tony Stewart first parti...,Age of 21,272,AGE,Is temporal,Age of 21,not defined
215,What was the age when Emma Miskew was appeared...,Age of 29,291,AGE,Is temporal,Age of 29,not defined
229,What was the age when Aude Gemma Billard was r...,Age of 45,321,AGE,Is temporal,Age of 45,not defined
301,What is the age of Steve Jobs when married Lau...,Age of 36,353,AGE,Is temporal,Age of 36,not defined


In [10]:
head_df = label_timeunit_year(
    head_df,
    head_df.query("answer.str.lower().str.match('age of \d+$')").index,
)

## Identify "year" time unit from question

In [11]:
head_df.query("answer_timeunit!='year'").loc[:, "question"].str.lower().str.split().apply(
    lambda x: tuple(x[:3])
).value_counts()

question
(how, many, years)    192
(how, old, was)        72
(when, was, the)       60
(what, year, did)      58
(how, many, days)      35
                     ... 
(when, was, ac/dc)      1
(when, did, ac/dc)      1
(how, many, year)       1
(on, what, date)        1
(when, did, sri)        1
Name: count, Length: 111, dtype: int64

### Questions starting with "How many years"

In [12]:
# Pick most frequent question indicating answer to have "year" time unit
# Exclude QA-pairs where answere is not just digits
head_df.query(
    "question.str.lower().str.startswith('how many years') "
    "and answer_timeunit!='year' " 
    "and ~answer.str.match('\d+')"
)

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit
1023,How many years passed between the PBY Catalina...,Nearly 22,1130,COUNT,Is temporal,Nearly 22,not defined
1094,How many years after did England won the T20 W...,Aftter 5 Years (2010),1172,TEMPORAL,Is temporal,Aftter 5 Years (2010),not defined


In [13]:
head_df = label_timeunit_year(
    head_df,
    head_df.query(
        "question.str.lower().str.startswith('how many years') "
        "and answer_timeunit!='year' "
        "and answer.str.match('^\d+$')"
    ).index,
)

### Questions starting with "How old was"

In [14]:
# Pick second most frequent question indicating answer to have "year" time unit
# Exclude QA-pairs where answere is not just digits
head_df.query(
    "question.str.lower().str.startswith('how old was') "
    "and answer_timeunit!='year' " 
    "and ~answer.str.match('\d+')"
)

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit


Old answers to questions starting with "How old was" contain only digits and can be assumed to have "answer" time unit.

In [15]:
head_df = label_timeunit_year(
    head_df,
    head_df.query(
        "question.str.lower().str.startswith('how old was') "
        "and answer_timeunit!='year' "
        "and answer.str.match('\d+')"
    ).index,
)

### Questions starting with "When was the"

In [16]:
# To keep things simple, let us only look for answers containing digits
# We are not excluding many QA-pairs this way
head_df.query(
        "question.str.lower().str.startswith('when was the') "
        "and answer_timeunit!='year' "
        "and ~answer.str.match('\d+')"
)

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit
553,When was the University of Cambridge was estab...,1209,606,TEMPORAL,Is temporal,1209,not defined
921,When was the first foreshock of the 1960 Agadi...,February 23,1036,TEMPORAL,Is temporal,February 23,not defined
927,When was the last show on the Better Day World...,"December 1, 2011",1054,TEMPORAL,Is temporal,"December 1, 2011",not defined
928,When was the first day you could catch a show ...,"July 17, 2011",1054,TEMPORAL,Is temporal,"July 17, 2011",not defined
960,"When was the first time that ""3 a.m. Eternal"" ...",May 1989,1071,TEMPORAL,Is temporal,May 1989,not defined
970,When was the the 49th Parallel released in the...,"April 15, 1942",1078,TEMPORAL,Is temporal,"April 15, 1942",not defined


In [17]:
head_df = label_timeunit_year(
    head_df,
    head_df.query(
        "question.str.lower().str.startswith('when was the') "
        "and answer_timeunit!='year' "
        "and answer.str.match('^\d+$')"
    ).index,
)

### Questions starting with "What year did"

In [18]:
# To keep things simple, let us only look for answers containing digits
# Actually, there are only answers containing only digits
head_df.query(
        "question.str.lower().str.startswith('what year did') "
        "and answer_timeunit!='year' "
        "and ~answer.str.match('\d+')"
)

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit


In [19]:
head_df = label_timeunit_year(
    head_df,
    head_df.query(
        "question.str.lower().str.startswith('what year did') "
        "and answer_timeunit!='year' "
        "and answer.str.match('\d+')"
    ).index,
)

In [20]:
head_df.query("answer_timeunit!='year'")

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit
5,For how many years had Benedict Cumberbatch be...,17,3,COUNT,Is temporal,17,not defined
11,For how many years has Benedict Cumberbatch be...,7,3,COUNT,Is temporal,7,not defined
22,How many years did Joan Crawford was active ca...,50 Years (1924–1974),8,TEMPORAL,Is temporal,50 Years (1924–1974),not defined
23,How many years did Joan Crawford and Alfred St...,4 Years (from 1955 to 1959),8,TEMPORAL,Is temporal,4 Years (from 1955 to 1959),not defined
24,How long did each of Joan Crawford's marriages...,4,8,UNKNOWN,Is temporal,four,not defined
...,...,...,...,...,...,...,...
1117,How long did it take for Ingenuity to go from ...,16 days,1206,TEMPORAL,Is temporal,16 days,not defined
1118,How long was it between the day of Ingenuity's...,2 months,1206,TEMPORAL,Is temporal,2 months,not defined
1120,How many days did Ingenuity take from date of ...,16 days,1206,TEMPORAL,Is temporal,16 days,not defined
1121,How much time since deployment did it take for...,1 year 2 months,1206,TEMPORAL,Is temporal,1 year 2 months,not defined


### Questions starting with "at what age"

In [21]:
# To keep things simple, let us only look for answers containing digits
# Actually, there are only answers containing only digits
head_df.query(
        "question.str.lower().str.startswith('at what age') "
        "and answer_timeunit!='year' "
        "and ~answer.str.match('\d+')"
)

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit


In [22]:
head_df = label_timeunit_year(
    head_df,
    head_df.query(
        "question.str.lower().str.startswith('at what age') "
        "and answer_timeunit!='year' "
        "and answer.str.match('\d+')"
    ).index,
)

### Question starting with "For how many years"

In [23]:
head_df.query(
    "answer_timeunit!='year'and question.str.lower().str.startswith('for how many years') "
)

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit
5,For how many years had Benedict Cumberbatch be...,17,3,COUNT,Is temporal,17,not defined
11,For how many years has Benedict Cumberbatch be...,7,3,COUNT,Is temporal,7,not defined
94,For how many years did James Worthy play for t...,12,81,COUNT,Is temporal,12,not defined
257,For how many years was Nicki Minaj with her se...,3,347,COUNT,Is temporal,3,not defined
363,For how many years did Benny McCoy play profes...,3,405,COUNT,Is temporal,3,not defined
403,For how many years did Black Dog Siding operat...,90,464,COUNT,Is temporal,90,not defined
431,For how many years did Draško Nenadić play wit...,3,492,COUNT,Is temporal,3,not defined
448,For how many years did Žarko Marković play for...,7,504,COUNT,Is temporal,7,not defined
452,For how many years did Roger Federer play pro...,6,512,COUNT,Is temporal,6,not defined
508,For how many years did Abdul Jeelani play prof...,13,561,COUNT,Is temporal,13,not defined


In [24]:
head_df = label_timeunit_year(
    head_df,
    head_df.query(
        "answer_timeunit!='year'and question.str.lower().str.startswith('for how many years') "
    ).index,
)

### Question containing "what year"

In [25]:
head_df.query("answer_timeunit!='year'and question.str.lower().str.contains('what year')")

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit
57,What year was Roy Emerson inducted into the In...,1982,50,TEMPORAL,Is temporal,1982,not defined
79,What year was Chuck Howley drafted into the NFL?,1958,74,TEMPORAL,Is temporal,1958,not defined
84,During what year did Manning receive the most ...,2003,75,TEMPORAL,Is temporal,2003,not defined
87,What year was Eli Manning drafted into the NFL?,2004,75,TEMPORAL,Is temporal,2004,not defined
112,What years did Carolina Albuquerque win medals...,"1999, 2005, 2006, 2008",93,TEMPORAL,Is temporal,"1999, 2005, 2006, 2008",not defined
122,In what year did Jager win the most overall Ol...,1988,118,TEMPORAL,Is temporal,1988,not defined
127,In what year did Bayinnaung first reign?,1565,119,TEMPORAL,Is temporal,1565,not defined
136,Since 2013 what year did Chen Yufei not win a ...,2021,129,TEMPORAL,Is temporal,2021,not defined
137,In what year did Chen Yufei win the most medals?,2016,129,TEMPORAL,Is temporal,2016,not defined
140,In what year did Shi Yuqi win the most gold me...,2014,166,TEMPORAL,Is temporal,2014,not defined


In [26]:
head_df = label_timeunit_year(
    head_df,
    head_df.query(
        "answer_timeunit!='year'and question.str.lower().str.contains('what year')"
    ).index,
)

## Remaining unlabeled QA-pairs

Shown below we see, that there are not many patterns left to exploit for labeling answers expecting a "year" time unit.

In [27]:
head_df.query(
    "answer_timeunit!='year'"
    "and ~answer.str.lower().str.contains('month') "
    "and ~question.str.lower().str.startswith('how many months') "
    "and ~answer.str.lower().str.contains('days') "
    "and ~answer.str.lower().str.contains('hour') "
    "and ~answer.str.lower().str.contains('minute') "
    "and ~answer.str.contains('\d{4}\)')"  # Answers containing a year (range) in parenthesis
).loc[:, "question"].str.lower().str.split().apply(lambda x: tuple(x[:3])).value_counts()

question
(what, was, the)      15
(when, was, the)      10
(what, is, the)        7
(when, did, the)       7
(how, many, days)      6
                      ..
(when, was, ac/dc)     1
(when, did, ac/dc)     1
(how, many, year)      1
(on, what, date)       1
(when, did, sri)       1
Name: count, Length: 92, dtype: int64

In [28]:
head_df.to_csv("data/notebook_output/03_label_answers_with_year_timeunit_head_data.csv", index=False)